In [ ]:
from pathlib import Path

SEG_EVAL_DIR = Path("C:\\Users\\juhe9\\repos\\MasterThesis\\ForkSight\\Data\\evaluation_output\\segmentation\\20260428_101251")
JD_EVAL_DIR = Path("C:\\Users\\juhe9\\repos\\MasterThesis\\ForkSight\\Data\\evaluation_output\\junction_detection\\20260428_114907")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

In [ ]:
def bold_best(df: pd.DataFrame, lower_is_better_cols: set[str] | None = None):
    """Return a Styler that bolds the best value in each column."""
    lower_is_better_cols = lower_is_better_cols or set()

    def _apply(s: pd.Series) -> list[str]:
        numeric = pd.to_numeric(s, errors="coerce")
        want_min = s.name in lower_is_better_cols
        best = numeric.min() if want_min else numeric.max()
        return [
            "font-weight: bold" if (not np.isnan(v) and v == best) else ""
            for v in numeric
        ]

    float_cols = df.select_dtypes(include="number").columns
    fmt = {c: "{:.4f}" for c in float_cols}
    return (
        df.style
        .apply(_apply, axis=0)
        .format(fmt, na_rep="–")
        .set_table_styles([{"selector": "th, td", "props": "font-family: inherit; font-size: inherit;"}])
    )


def horizontal_bar_groups(df: pd.DataFrame, col_groups: list[tuple],
                          lower_is_better: set[str] | None = None,
                          col_rename: dict | None = None,
                          ylim: tuple | None = (0, 1.18),
                          cols_per_row: int | None = None,
                          suptitle: str = "") -> None:
    """Bar chart grid, one subplot per column group.  Legend is placed above the figure.

    Parameters
    ----------
    ylim         : (ymin, ymax) or None to auto-scale each subplot independently.
    cols_per_row : max subplots per row; None keeps all groups in a single row.
    """
    from matplotlib.gridspec import GridSpec

    lower_is_better = lower_is_better or set()
    col_rename = col_rename or {}

    filtered_groups = [
        [c for c in group if c in df.columns]
        for group in col_groups
    ]
    filtered_groups = [g for g in filtered_groups if g]
    if not filtered_groups:
        print("  (no columns available for bar chart)")
        return

    model_labels = [str(n) for n in df.index]
    total_cols = sum(len(g) for g in filtered_groups)
    colors = plt.cm.tab10(np.linspace(0, 0.9, len(df)))

    # Split groups into rows
    cpr = cols_per_row or len(filtered_groups)
    row_chunks = [filtered_groups[i:i + cpr]
                  for i in range(0, len(filtered_groups), cpr)]
    n_rows = len(row_chunks)

    ncol = min(len(df), 4)
    n_legend_rows = (len(df) + ncol - 1) // ncol
    top_gap = 0.05 * n_legend_rows + (0.07 if suptitle else 0.03)
    subplot_top = 1.2 - top_gap

    row_height = max(4, len(df) * 0.5 + 1.5)
    fig_width = max(10, 2.0 * total_cols)
    fig_height = row_height * n_rows

    fig = plt.figure(figsize=(fig_width, fig_height))
    outer_gs = GridSpec(n_rows, 1, figure=fig, hspace=0.45)

    first_handles, first_labels = None, None

    for row_idx, chunk in enumerate(row_chunks):
        width_ratios = [len(g) for g in chunk]
        inner_gs = outer_gs[row_idx].subgridspec(
            1, len(chunk), width_ratios=width_ratios, wspace=0.3)

        for ci, cols in enumerate(chunk):
            ax = fig.add_subplot(inner_gs[0, ci])
            x = np.arange(len(cols))
            width = 0.8 / max(len(df), 1)

            bars_by_col: dict[int, list[tuple]] = {k: [] for k in range(len(cols))}

            for i, (model, row) in enumerate(df.iterrows()):
                vals = [float(row.get(c, np.nan)) for c in cols]
                offset = (i - len(df) / 2 + 0.5) * width
                rects = ax.bar(x + offset, vals, width * 0.9,
                               label=model_labels[i], color=colors[i])
                for k, (rect, val) in enumerate(zip(rects, vals)):
                    bars_by_col[k].append((val, rect))

            if first_handles is None:
                first_handles, first_labels = ax.get_legend_handles_labels()

            # Annotation offset relative to data range
            all_vals = [v for entries in bars_by_col.values()
                        for v, _ in entries if not np.isnan(v)]
            y_range = (max(all_vals) - min(all_vals)) if len(all_vals) >= 2 else 1.0
            ann_offset = y_range * 0.025

            for k, col in enumerate(cols):
                entries = [(v, r) for v, r in bars_by_col[k] if not np.isnan(v)]
                if not entries:
                    continue
                want_min = col in lower_is_better
                best_val, best_rect = (min(entries, key=lambda t: t[0]) if want_min
                                       else max(entries, key=lambda t: t[0]))
                bar_x = best_rect.get_x() + best_rect.get_width() / 2
                ax.text(bar_x, best_rect.get_height() + ann_offset,
                        f"{best_val:.3f}",
                        ha="center", va="bottom", fontsize=7, fontweight="bold")

            tick_labels = [col_rename.get(c, c).replace(" ", "\n") for c in cols]
            ax.set_xticks(x)
            ax.set_xticklabels(tick_labels, fontsize=8)
            ax.grid(axis="y", linestyle="--", alpha=0.4)

            if ylim is not None:
                ax.set_ylim(*ylim)
            else:
                if all_vals:
                    ax.set_ylim(0, max(all_vals) + y_range * 0.15)

    plt.tight_layout(rect=[0, 0, 1, subplot_top])
    fig.legend(
        first_handles, first_labels,
        loc="lower center",
        bbox_to_anchor=(0.5, subplot_top),
        ncol=ncol,
        fontsize=8,
        framealpha=0.8,
    )
    if suptitle:
        fig.suptitle(suptitle, fontsize=12, y=0.99)

    plt.show()
    plt.close()

In [ ]:
# Load segmentation metrics
_seg_csv = SEG_EVAL_DIR / "csv" / "metrics.csv"
if _seg_csv is None:
    raise FileNotFoundError(
        f"metrics.csv not found in {SEG_EVAL_DIR} or {SEG_EVAL_DIR / 'csv'}")

df_seg = pd.read_csv(_seg_csv, index_col=0)
df_seg.index.name = "Model"
print(f"Loaded segmentation metrics: {len(df_seg)} model(s) from {_seg_csv}")
df_seg.index = df_seg.index.map(str)  # ensure string index

# Optional: persistence distances
_dist_csv = SEG_EVAL_DIR / "csv" / "persistence_distances.csv"
df_dist = pd.read_csv(_dist_csv) if _dist_csv else pd.DataFrame()
if not df_dist.empty:
    print(f"Loaded persistence distances: {len(df_dist)} row(s)")

### Segmentation Metrics — Raw

In [ ]:
SEG_RAW_COLS = ["Dice", "IoU", "clDice", "tprec", "tsens"]

_raw_cols = [c for c in SEG_RAW_COLS if c in df_seg.columns]
if _raw_cols:
    display(bold_best(df_seg[_raw_cols],))
else:
    print("No raw segmentation columns found.")

### Segmentation Metrics — Post-processed

In [ ]:
SEG_PP_COLS = [
    "Dice Postprocessed", "IoU Postprocessed", "clDice Postprocessed",
    "tprec Postprocessed", "tsens Postprocessed",
]

_pp_cols = [c for c in SEG_PP_COLS if c in df_seg.columns]
if _pp_cols:
    display(bold_best(df_seg[_pp_cols]))
else:
    print("No post-processed segmentation columns found.")

### Segmentation Metrics — Topological (Persistence Distances)

Lower is better for all distance metrics.

In [ ]:
SEG_TOPO_COLS = [
    "Wasserstein B0 Raw", "Bottleneck B0 Raw",
    "Wasserstein B1 Raw", "Bottleneck B1 Raw",
    "Wasserstein B0 SDT", "Bottleneck B0 SDT",
    "Wasserstein B1 SDT", "Bottleneck B1 SDT",
]

_topo_cols = [c for c in SEG_TOPO_COLS if c in df_seg.columns]
if _topo_cols:
    display(bold_best(df_seg[_topo_cols], lower_is_better_cols=set(_topo_cols)))
else:
    print("No topological metric columns found.")

### Segmentation Bar Charts

In [ ]:
_SEG_RENAME = {
    "tprec": "Skeleton\nPrecision",
    "tsens": "Skeleton Sensitivity\n(Recall)",
    "tprec Postprocessed": "Skeleton\nPrecision",
    "tsens Postprocessed": "Skeleton Sensitivity\n(Recall)",
}

horizontal_bar_groups(
    df_seg,
    col_groups=[("Dice", "IoU", "clDice", "tprec", "tsens")],
    col_rename=_SEG_RENAME,
    suptitle="Segmentation — Raw",
)

horizontal_bar_groups(
    df_seg,
    col_groups=[("Dice Postprocessed", "IoU Postprocessed", "clDice Postprocessed",
                 "tprec Postprocessed", "tsens Postprocessed")],
    col_rename=_SEG_RENAME,
    suptitle="Segmentation — Post-processed",
)

In [ ]:
# ── Combined ablation grid: all four metrics × two ablations ─────────────────
_ABL_GROUP1 = [
    "SAM_LoRA_Ablation_100_0",
    "SAM_LoRA_Ablation_75_25",
    "SAM_LoRA_Ablation_50_50",
    "SAM_LoRA_Ablation_25_75",
]
_ABL_GROUP2 = [
    "SAM_LoRA_Ablation_100_0",
    "SAM_LoRA_Ablation_100_100",
    "SAM_LoRA_Ablation_100_200",
    "SAM_LoRA_Ablation_100_400",
    "sweep-cldice-lora-BEST",
]
_ABL_GROUP_LABELS = [
    "Dataset Diversity Ablation",
    "Dataset Size Ablation",
]

# (column, y-label, decimals, force_ymin, force_ymax)
_ABL_ALL_METRICS = [
    ("Dice Postprocessed",   "Dice",                    2, 0.860, None),
    ("clDice Postprocessed", "clDice",                  3, None,  0.980),
    ("Wasserstein B0 Raw",   r"$W_{H_0}$ (Prob.)",      2, 0.7,   1.5),
    ("Wasserstein B1 Raw",   r"$W_{H_1}$ (Prob.)",      2, 0.1,   0.3),
]

_FIG_DIR = Path("figures")
_FIG_DIR.mkdir(exist_ok=True)

_AUG_MULT = {"100": r"$1\times$", "200": r"$2\times$", "400": r"$4\times$"}

def _abl_label(name):
    if name == "sweep-cldice-lora-BEST":
        return "Full Dataset"
    if name.startswith("SAM_LoRA_Ablation_"):
        parts = name.removeprefix("SAM_LoRA_Ablation_").split("_")
        if len(parts) == 2:
            a, b = parts
            if b == "0":
                return "Raw Only"
            if a == "100" and b in _AUG_MULT:
                return f"Raw + {_AUG_MULT[b]}"
        return "/".join(parts)
    return name

FONT = 12
N_TICKS = 3  # number of y-ticks/gridlines per plot

n_rows = len(_ABL_ALL_METRICS)
n_cols = 2

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(8, 1.5 * n_rows + 0.6),
    sharex="col",
    sharey="row",
)

groups = [_ABL_GROUP1, _ABL_GROUP2]

for row, (metric_col, metric_label, decimals, force_ymin, force_ymax) in enumerate(_ABL_ALL_METRICS):
    # Compute y-range for this row using "nice" rounding
    all_vals = pd.concat([
        df_seg.loc[[m for m in g if m in df_seg.index], metric_col]
        for g in groups
    ]).astype(float)

    span = all_vals.max() - all_vals.min()
    raw_step = span / (N_TICKS - 1)
    magnitude = 10 ** np.floor(np.log10(raw_step))
    for nice in (1, 2, 2.5, 5, 10):
        step = nice * magnitude
        if step >= raw_step:
            break

    ymin = np.floor(all_vals.min() / step) * step
    ymax = np.ceil(all_vals.max() / step) * step
    if force_ymin is not None:
        ymin = min(ymin, force_ymin)
    if force_ymax is not None:
        ymax = max(ymax, force_ymax)

    # Force exactly N_TICKS evenly spaced ticks across the final range
    inner = np.round(np.linspace(ymin, ymax, N_TICKS + 2)[1:-1], decimals + 1)
    #yticks = np.round(np.concatenate([[ymin], inner, [ymax]]), decimals + 1)

    for col, (group, group_label) in enumerate(zip(groups, _ABL_GROUP_LABELS)):
        ax = axes[row, col]
        present = [m for m in group if m in df_seg.index]
        vals = df_seg.loc[present, metric_col].astype(float).values
        n = len(present)
        x = np.arange(n)
        labels = [_abl_label(m) for m in present]

        ax.plot(x, vals, marker="o", linewidth=1.0, markersize=4, zorder=3)

        for xi, val in zip(x, vals):
            ax.annotate(f"{val:.3f}", xy=(xi, val),
                        xytext=(0, 6), textcoords="offset points",
                        ha="center", va="bottom", fontsize=FONT - 3)

        ax.set_xticks(x)
        ax.set_xlim(-0.4, n - 0.6)
        ax.tick_params(axis="y", labelsize=FONT - 1)
        ax.grid(axis="y", linestyle="--", alpha=0.4)
        #ax.set_ylim(ymin, ymax)
        #ax.set_yticks(yticks)
        ax.set_ylim(ymin, ymax)
        ax.set_yticks([ymin, ymax])              # major ticks at borders → labeled
        ax.set_yticks(inner, minor=True)

        ax.grid(axis="y", which="major", linestyle="--", alpha=0.4)
        ax.grid(axis="y", which="minor", linestyle="--", alpha=0.4)

        # Hide y tick labels on the right column (sharey aligns the gridlines)
        if col == 1:
            ax.tick_params(axis="y", which="both", left=False, labelleft=False)

        # Column titles only on the top row
        if row == 0:
            ax.set_title(group_label, fontsize=FONT + 1, pad=4)

        # Y-axis label only on the left column
        if col == 0:
            ax.set_ylabel(metric_label, fontsize=FONT)

        # X-tick labels only on the bottom row
        if row == n_rows - 1:
            ax.set_xticklabels(labels, rotation=15, ha="right", fontsize=FONT)
        else:
            ax.set_xticklabels([])

plt.tight_layout(pad=1.0)
plt.subplots_adjust(wspace=0.05, hspace=0.2)
fig.savefig(_FIG_DIR / "ablation_combined.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

In [ ]:
_TOPO_LOWER = set(SEG_TOPO_COLS)  # all distance metrics: lower is better

for _suffix, _label in [("Raw", "Raw predictions"), ("SDT", "Signed distance transform")]:
    horizontal_bar_groups(
        df_seg,
        col_groups=[
            (f"Wasserstein B0 {_suffix}",),
            (f"Wasserstein B1 {_suffix}",),
            (f"Bottleneck B0 {_suffix}",),
            (f"Bottleneck B1 {_suffix}",),
        ],
        lower_is_better=_TOPO_LOWER,
        col_rename={
            f"Wasserstein B0 {_suffix}": "Wasserstein\nB0",
            f"Wasserstein B1 {_suffix}": "Wasserstein\nB1",
            f"Bottleneck B0 {_suffix}":  "Bottleneck\nB0",
            f"Bottleneck B1 {_suffix}":  "Bottleneck\nB1",
        },
        ylim=None,
        suptitle=f"Topological distances — {_label} (lower is better)",
    )

### Persistence Diagrams

Birth–death scatter plots from the raw and SDT persistence CSVs.

In [ ]:
for filename, title in [
    ("persistence_raw_b0.csv", "Raw — Betti 0 (connected components)"),
    ("persistence_raw_b1.csv", "Raw — Betti 1 (loops)"),
    ("persistence_sdt_b0.csv", "SDT — Betti 0"),
    ("persistence_sdt_b1.csv", "SDT — Betti 1"),
]:
    _csv = SEG_EVAL_DIR / "csv" / filename
    if _csv is None:
        print(f"  {filename} not found, skipping.")
        continue

    df_pd = pd.read_csv(_csv)
    models = df_pd["model"].unique() if "model" in df_pd.columns else []
    if len(models) == 0:
        print(f"  {filename}: no model column, skipping.")
        continue

    colors = plt.cm.tab10(np.linspace(0, 0.9, len(models)))
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    for ax, ptype in zip(axes, ["predicted", "groundtruth"]):
        df_sub = df_pd[df_pd["type"] ==
                       ptype] if "type" in df_pd.columns else df_pd
        for i, model in enumerate(models):
            df_m = df_sub[df_sub["model"] ==
                          model] if "model" in df_sub.columns else df_sub
            if df_m.empty:
                continue
            b, d = df_m["birth"], df_m["death"]
            ax.scatter(b, d, s=4, alpha=0.4, color=colors[i],
                       label=str(model)[-25:])
        # Diagonal (birth = death)
        lim = max(ax.get_xlim()[1], ax.get_ylim()[1])
        ax.plot([0, lim], [0, lim], "k--", linewidth=0.8, alpha=0.5)
        ax.set_xlabel("Birth")
        ax.set_ylabel("Death")
        ax.set_title(ptype.capitalize())
        if ptype == "predicted":
            ax.legend(markerscale=3, fontsize=7, loc="lower right")

    fig.suptitle(f"Persistence diagram — {title}", fontsize=11)
    plt.tight_layout()
    plt.show()
    plt.close()

---
## Junction Detection Evaluation

In [ ]:
# ── Load junction detection metrics ────────────────────────────────────────
_jd_csv = JD_EVAL_DIR / "metrics.csv"
if _jd_csv is None:
    raise FileNotFoundError(
        f"metrics.csv not found in {JD_EVAL_DIR} or {JD_EVAL_DIR / 'csv'}")

df_jd = pd.read_csv(_jd_csv, index_col="model")
print(
    f"Loaded junction detection metrics: {len(df_jd)} model(s) from {_jd_csv}")
df_jd.index = df_jd.index.map(str)

In [ ]:
# Ground truth junction counts (fixed test set — should be identical across all models)
_gt_3way = df_jd["pp_gt_3way_total"].astype(int)
_gt_4way = df_jd["pp_gt_4way_total"].astype(int)

assert _gt_3way.nunique() == 1, "GT 3-way total differs across models"
assert _gt_4way.nunique() == 1, "GT 4-way total differs across models"

n_gt_3way = int(_gt_3way.iloc[0])
n_gt_4way = int(_gt_4way.iloc[0])
n_gt_total = n_gt_3way + n_gt_4way

print(f"Ground truth junctions in test set:")
print(f"  3-way : {n_gt_3way}")
print(f"  4-way : {n_gt_4way}")
print(f"  Total : {n_gt_total}")

In [ ]:
# Derive micro-averaged combined joint metrics from the per-class counts already in the CSV.
# (Avoids re-running the slow compute script.)
def _prf(tp, fp, fn):
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return prec, rec, f1

for _pfx in ("raw_", "pp_"):
    tp = df_jd[f"{_pfx}class_tp_3way"] + df_jd[f"{_pfx}class_tp_4way"]
    fp = df_jd[f"{_pfx}class_fp_3way"] + df_jd[f"{_pfx}class_fp_4way"]
    fn = df_jd[f"{_pfx}class_fn_3way"] + df_jd[f"{_pfx}class_fn_4way"]
    prf = [_prf(tp.iloc[i], fp.iloc[i], fn.iloc[i]) for i in range(len(df_jd))]
    df_jd[f"{_pfx}class_tp_combined"]        = tp.values
    df_jd[f"{_pfx}class_fp_combined"]        = fp.values
    df_jd[f"{_pfx}class_fn_combined"]        = fn.values
    df_jd[f"{_pfx}class_precision_combined"] = [p for p, _, _ in prf]
    df_jd[f"{_pfx}class_recall_combined"]    = [r for _, r, _ in prf]
    df_jd[f"{_pfx}class_f1_combined"]        = [f for _, _, f in prf]

In [ ]:
# Column groups — all higher-is-better
_JD_LOCALISATION      = ["precision_loc", "recall_loc", "f1_loc"]
_JD_DETECTION_RECALL  = ["detection_recall_3way", "detection_recall_4way"]
_JD_TYPE_OVERALL      = ["type_accuracy"]
_JD_CLASS_COMBINED    = ["class_precision_combined", "class_recall_combined", "class_f1_combined"]
_JD_CLASS_3WAY        = ["class_precision_3way", "class_recall_3way", "class_f1_3way"]
_JD_CLASS_4WAY        = ["class_precision_4way", "class_recall_4way", "class_f1_4way"]

_JD_DISPLAY_COLS = (
    _JD_LOCALISATION + _JD_DETECTION_RECALL + _JD_TYPE_OVERALL
    + _JD_CLASS_COMBINED + _JD_CLASS_3WAY + _JD_CLASS_4WAY
)

_DISPLAY_RENAME = {
    "precision_loc":            "Precision (loc)",
    "recall_loc":               "Recall (loc)",
    "f1_loc":                   "F1 (loc)",
    "detection_recall_3way":    "Det. Recall 3-way",
    "detection_recall_4way":    "Det. Recall 4-way",
    "type_accuracy":            "Type Accuracy",
    "class_precision_combined": "Precision (joint)",
    "class_recall_combined":    "Recall (joint)",
    "class_f1_combined":        "F1 (joint)",
    "class_precision_3way":     "Precision 3-way",
    "class_recall_3way":        "Recall 3-way",
    "class_f1_3way":            "F1 3-way",
    "class_precision_4way":     "Precision 4-way",
    "class_recall_4way":        "Recall 4-way",
    "class_f1_4way":            "F1 4-way",
}


def _jd_table(df: pd.DataFrame, prefix: str, title: str) -> None:
    cols_in_df = [f"{prefix}{c}" for c in _JD_DISPLAY_COLS
                  if f"{prefix}{c}" in df.columns]
    if not cols_in_df:
        print(f"No '{prefix}' columns found.")
        return
    display(HTML(f"<h4>{title}</h4>"))
    sub = df[cols_in_df].copy()
    sub.columns = [_DISPLAY_RENAME.get(c.removeprefix(prefix), c) for c in cols_in_df]
    display(bold_best(sub))

**Precision / Recall / F1 (loc):** class-agnostic; a prediction is a TP if it falls within the pixel threshold of *any* GT junction, regardless of type.
**Det. Recall 3-way / 4-way:** per-GT-class localisation recall — of all GT junctions of that type, what fraction were spatially matched by *any* prediction (regardless of predicted type). Also shown in the confusion matrix title.
**Type Accuracy:** among spatially-matched TPs only — fraction whose type label (3-way / 4-way) matches GT.
**Precision / Recall / F1 (joint):** micro-averaged over both classes; a prediction is a TP for class $c$ only if it is both correctly localised *and* correctly labelled — equivalent to summing the per-class TP/FP/FN counts before computing the metrics.
**Precision / Recall / F1 3-way & 4-way:** same joint criterion, broken out per class. FP and FN include both spurious/missed detections and type misclassifications (see confusion matrices below).

### Junction Detection Metrics — Raw

In [ ]:
_jd_table(df_jd, prefix="raw_", title="Raw (no postprocessing)")

### Junction Detection Metrics — Post-processed

In [ ]:
_jd_table(df_jd, prefix="pp_", title="Post-processed")

### Junction Detection Bar Charts

In [ ]:
for _prefix, _label in [("raw_", "Raw"), ("pp_", "Post-processed")]:
    horizontal_bar_groups(
        df_jd,
        col_groups=[
            tuple(f"{_prefix}{c}" for c in _JD_LOCALISATION),
            tuple(f"{_prefix}{c}" for c in _JD_DETECTION_RECALL),
            tuple(f"{_prefix}{c}" for c in _JD_TYPE_OVERALL + _JD_CLASS_COMBINED),
            tuple(f"{_prefix}{c}" for c in _JD_CLASS_3WAY),
            tuple(f"{_prefix}{c}" for c in _JD_CLASS_4WAY),
        ],
        cols_per_row=2,
        suptitle=f"Junction Detection — {_label}",
    )

### Confusion Matrices

Displayed from the PNG files saved by `compute_metrics_junction_detection.py` (post-processed results).

In [ ]:
import string

_CM_FONT = 11

_CM_MODELS = {
    "sweep-cldice-BEST":                                              "M1 (SAM, Img. Enc. Frozen, clDice)",
    "SAM_LoRA_BCE_Dice_Only":                                         "M2 (SAM, Img. Enc. LoRA, Baseline)",
    "sweep-cldice-lora-BEST":                                         "M3 (SAM, Img. Enc. LoRA, clDice)",
    "nnunet/Dataset001_Segmentation_v1/nnUNetTrainerClDiceLoss":      "M4 (U-Net, clDice)",
}

_CM_COL_LABELS = ["Pred 3-way", "Pred 4-way", "Not found\n(FN)"]
_CM_ROW_LABELS = ["GT 3-way", "GT 4-way", "No GT\n(FP)"]


def _plot_cm_from_row(ax, row, title, prefix="pp_",
                      show_x_labels=True, show_y_labels=True):
    cm = np.array([
        [row[f"{prefix}cm_gt3_pred3"], row[f"{prefix}cm_gt3_pred4"], row[f"{prefix}fn_3way"]],
        [row[f"{prefix}cm_gt4_pred3"], row[f"{prefix}cm_gt4_pred4"], row[f"{prefix}fn_4way"]],
        [row[f"{prefix}fp_3way"],      row[f"{prefix}fp_4way"],      float("nan")],
    ], dtype=float)

    row_totals = np.array([
        row[f"{prefix}gt_3way_total"],
        row[f"{prefix}gt_4way_total"],
        row[f"{prefix}fp_3way"] + row[f"{prefix}fp_4way"],
    ], dtype=float)

    # Color by row-normalized percentage -> shared, meaningful 0-100% scale
    with np.errstate(divide="ignore", invalid="ignore"):
        cm_pct = np.where(row_totals[:, None] > 0,
                          100.0 * cm / row_totals[:, None],
                          np.nan)

    masked = np.ma.masked_invalid(cm_pct)
    im = ax.imshow(masked, cmap="Blues", vmin=0, vmax=100, aspect="equal")

    # Greyed-out N/A cell
    ax.add_patch(plt.Rectangle((1.5, 1.5), 1, 1, color="#cccccc", zorder=2))
    ax.text(2, 2, "N/A", ha="center", va="center",
            fontsize=_CM_FONT, color="#666666", zorder=3)

    ax.set_xticks([0, 1, 2])
    ax.set_yticks([0, 1, 2])
    ax.set_xticklabels(_CM_COL_LABELS if show_x_labels else [""] * 3,
                       fontsize=_CM_FONT)
    ax.set_yticklabels(_CM_ROW_LABELS if show_y_labels else [""] * 3,
                       fontsize=_CM_FONT)
    if show_x_labels:
        ax.set_xlabel("Prediction", fontsize=_CM_FONT)
    if show_y_labels:
        ax.set_ylabel("Ground truth", fontsize=_CM_FONT)

    for r in range(3):
        for c in range(3):
            if r == 2 and c == 2:
                continue
            val = cm[r, c]
            if np.isnan(val):
                continue
            pct = cm_pct[r, c]
            pct_str = f"\n({pct:.0f}%)" if not np.isnan(pct) else ""
            text_color = "white" if (not np.isnan(pct) and pct > 60) else "black"
            ax.text(c, r, f"{int(val)}{pct_str}",
                    ha="center", va="center",
                    fontsize=_CM_FONT, color=text_color,
                    fontweight="bold", zorder=4)

    ax.axhline(1.5, color="black", linewidth=1.5, linestyle="--")
    ax.axvline(1.5, color="black", linewidth=1.5, linestyle="--")
    ax.set_title(title, fontsize=_CM_FONT)
    return im


_cm_rows = [(title, df_jd.loc[key]) for key, title in _CM_MODELS.items()
            if key in df_jd.index]

_cols = 2
_rows = (len(_cm_rows) + _cols - 1) // _cols
fig, axes = plt.subplots(_rows, _cols,
                         figsize=(4.2 * _cols, 4.2 * _rows),
                         squeeze=False)

im = None
for i, (title, row) in enumerate(_cm_rows):
    r, c = i // _cols, i % _cols
    ax = axes[r][c]
    letter = string.ascii_lowercase[i]
    im = _plot_cm_from_row(
        ax, row, f"({letter}) {title}", prefix="pp_",
        show_x_labels=(r == _rows - 1),
        show_y_labels=(c == 0),
    )

for ax in axes.flat[len(_cm_rows):]:
    ax.set_visible(False)

#plt.suptitle("Junction Detection — Confusion Matrices (post-processed)",
#             fontsize=_CM_FONT + 2, y=1.01)

plt.subplots_adjust(wspace=0.05, hspace=0.15, right=0.91)

# Single shared colorbar on the right
cbar_ax = fig.add_axes([0.93, 0.15, 0.018, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label("% of row total", fontsize=_CM_FONT)
cbar.ax.tick_params(labelsize=_CM_FONT - 1)

fig.savefig(_FIG_DIR / "results_exp3_confusion_matrices.png", dpi=300, bbox_inches="tight")

plt.show()
plt.close()

In [ ]:
# Display the pre-saved PNG confusion matrices for comparison
from PIL import Image as _PILImage

_cm_dir = JD_EVAL_DIR / "confusion_matrices"
_cm_pngs = sorted(_cm_dir.glob("cm_*.png")) if _cm_dir.is_dir() else []

if not _cm_pngs:
    print("No confusion-matrix PNGs found.")
else:
    n = len(_cm_pngs)
    _cols = 2
    _rows = (n + _cols - 1) // _cols
    fig, axes = plt.subplots(_rows, _cols, figsize=(6 * _cols, 5.5 * _rows))
    axes = np.array(axes).flatten()
    for ax, png in zip(axes, _cm_pngs):
        ax.imshow(np.array(_PILImage.open(png)))
        ax.axis("off")
        ax.set_title(png.stem.removeprefix("cm_"), fontsize=8)
    for ax in axes[n:]:
        ax.set_visible(False)
    plt.suptitle("Pre-saved PNGs (for comparison)", fontsize=11)
    plt.tight_layout()
    plt.show()
    plt.close()

### Raw vs Post-processed: Key Metrics Side-by-side

In [ ]:
_KEY_METRICS = ["f1_loc", "class_f1_combined", "class_f1_3way", "class_f1_4way"]
_short_labels = [str(n)[-30:] for n in df_jd.index]
colors = plt.cm.tab10(np.linspace(0, 0.9, len(df_jd)))

_present = [m for m in _KEY_METRICS
            if f"raw_{m}" in df_jd.columns and f"pp_{m}" in df_jd.columns]

if _present:
    fig, axes = plt.subplots(1, len(_present), figsize=(4 * len(_present), max(4, len(df_jd) * 0.5 + 1.5)),
                             squeeze=False)
    axes = axes[0]
    x = np.arange(len(df_jd))
    width = 0.35

    for ax, metric in zip(axes, _present):
        raw_vals = df_jd[f"raw_{metric}"].astype(float).values
        pp_vals  = df_jd[f"pp_{metric}"].astype(float).values
        ax.bar(x - width / 2, raw_vals, width, label="Raw", color="steelblue", alpha=0.8)
        ax.bar(x + width / 2, pp_vals,  width, label="Post-proc", color="darkorange", alpha=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(_short_labels, rotation=45, ha="right", fontsize=7)
        ax.set_ylim(0, 1.15)
        ax.set_title(_DISPLAY_RENAME.get(metric, metric), fontsize=9)
        ax.grid(axis="y", linestyle="--", alpha=0.4)
        ax.legend(fontsize=7)

    fig.suptitle("Raw vs Post-processed — Key Junction Detection Metrics", fontsize=11)
    plt.tight_layout()
    plt.show()
    plt.close()
else:
    print("Key metric columns not found in dataframe.")

---
## Segmentation ↔ Junction Detection Correlation Analysis

Spearman correlation coefficients between segmentation metrics (x) and post-processed junction detection metrics (y), plus scatter plots with OLS trend lines. Each data point is one model (n=17).

In [ ]:
from scipy import stats as _scipy_stats
import matplotlib.cm as _mcm
import matplotlib.colors as _mcolors

# ── Metric definitions ────────────────────────────────────────────────────────
SEG_CORR_COLS = {
    "Dice":                "Dice",
    "clDice":              "clDice",
    "tprec":               r"$T_{\text{prec}}$",
    "tsens":               r"$T_{\text{sens}}$",
    "Wasserstein B0 Raw":  r"$W_{H_0}$ (Prob.)",
    "Bottleneck B0 Raw":   r"$d_{B,H_0}$ (Prob.)",
    "Wasserstein B1 Raw":  r"$W_{H_1}$ (Prob.)",
    "Bottleneck B1 Raw":   r"$d_{B,H_1}$ (Prob.)",
    "Wasserstein B0 SDT":  r"$W_{H_0}$ (SDT)",
    "Bottleneck B0 SDT":   r"$d_{B,H_0}$ (SDT)",
    "Wasserstein B1 SDT":  r"$W_{H_1}$ (SDT)",
    "Bottleneck B1 SDT":   r"$d_{B,H_1}$ (SDT)",
}
SEG_LOWER_BETTER = {
    "Wasserstein B0 Raw", "Bottleneck B0 Raw",
    "Wasserstein B1 Raw", "Bottleneck B1 Raw",
    "Wasserstein B0 SDT", "Bottleneck B0 SDT",
    "Wasserstein B1 SDT", "Bottleneck B1 SDT",
}

JD_LOC_COLS = {
    "pp_precision_loc":            "Precision (loc)",
    "pp_recall_loc":               "Recall (loc)",
}
JD_DET_RECALL_COLS = {
    "pp_detection_recall_3way":    "Det. Recall 3-way",
    "pp_detection_recall_4way":    "Det. Recall 4-way",
}
JD_JOINT_COMBINED_COLS = {
    "pp_class_precision_combined": "Precision (joint)",
    "pp_class_recall_combined":    "Recall (joint)",
}
JD_JOINT_3WAY_COLS = {
    "pp_class_precision_3way":     "Precision 3-way",
    "pp_class_recall_3way":        "Recall 3-way",
}
JD_JOINT_4WAY_COLS = {
    "pp_class_precision_4way":     "Precision 4-way",
    "pp_class_recall_4way":        "Recall 4-way",
}
ALL_JD_CORR_COLS = {
    **JD_LOC_COLS,
    **JD_DET_RECALL_COLS,
    **JD_JOINT_COMBINED_COLS,
    **JD_JOINT_3WAY_COLS,
    **JD_JOINT_4WAY_COLS,
}

# ── Models to exclude from correlation computation ───────────────────────────
# (df_merged and the scatter plots still include all models)
_CORR_IGNORE_MODELS = [
    #"sweep-cldice-BEST",
    #"sweep-skeleton-recall-BEST",
]

# ── Merge: select only needed columns to avoid column-name conflicts ─────────
_seg_keep = [c for c in SEG_CORR_COLS if c in df_seg.columns]
_jd_keep  = [c for c in ALL_JD_CORR_COLS if c in df_jd.columns]
df_merged = df_seg[_seg_keep].join(df_jd[_jd_keep], how="inner")
print(f"Merged dataset: {len(df_merged)} models, "
      f"{len(_seg_keep)} seg cols, {len(_jd_keep)} JD cols")

# Filtered view used only for correlation computation
_df_corr_input = df_merged.drop(
    index=[m for m in _CORR_IGNORE_MODELS if m in df_merged.index]
)
if _CORR_IGNORE_MODELS:
    print(f"Correlation computed on {len(_df_corr_input)} models "
          f"(ignoring: {', '.join(_CORR_IGNORE_MODELS)})")

# ── Spearman correlation computation ─────────────────────────────────────────
_corr_rows = {}
for seg_col, seg_label in SEG_CORR_COLS.items():
    if seg_col not in _df_corr_input.columns:
        continue
    _corr_rows[seg_label] = {}
    for jd_col, jd_label in ALL_JD_CORR_COLS.items():
        if jd_col not in _df_corr_input.columns:
            _corr_rows[seg_label][jd_label] = np.nan
            continue
        valid = _df_corr_input[[seg_col, jd_col]].dropna()
        if len(valid) >= 3:
            r, _ = _scipy_stats.spearmanr(valid[seg_col].astype(float),
                                          valid[jd_col].astype(float))
        else:
            r = np.nan
        _corr_rows[seg_label][jd_label] = r

df_corr = pd.DataFrame(_corr_rows).T   # rows = seg metric, cols = JD metric

# ── Notebook display: continuous heatmap, no threshold ───────────────────────
df_corr_str = pd.DataFrame(
    {jd: {seg: ("–" if pd.isna(df_corr.loc[seg, jd])
                else f"{df_corr.loc[seg, jd]:+.3f}")
          for seg in df_corr.index}
     for jd in df_corr.columns}
)

# Plain-text labels for notebook (Styler doesn't render LaTeX math)
_NOTEBOOK_LABELS = {
    r"$T_{\text{prec}}$":    "T_prec",
    r"$T_{\text{sens}}$":    "T_sens",
    r"$W_{H_0}$ (Prob.)":    "W_H0 (Prob.)",
    r"$d_{B,H_0}$ (Prob.)":  "d_B,H0 (Prob.)",
    r"$W_{H_1}$ (Prob.)":    "W_H1 (Prob.)",
    r"$d_{B,H_1}$ (Prob.)":  "d_B,H1 (Prob.)",
    r"$W_{H_0}$ (SDT)":      "W_H0 (SDT)",
    r"$d_{B,H_0}$ (SDT)":    "d_B,H0 (SDT)",
    r"$W_{H_1}$ (SDT)":      "W_H1 (SDT)",
    r"$d_{B,H_1}$ (SDT)":    "d_B,H1 (SDT)",
}
df_corr_str_display = df_corr_str.rename(index=_NOTEBOOK_LABELS)

_cmap = _mcm.get_cmap("RdBu")

def _cell_bg(val_str):
    try:
        v = float(val_str)
    except (ValueError, TypeError):
        return "background-color: white"
    rgba = _cmap((v + 1) / 2)   # map [-1, 1] -> [0, 1]
    hex_color = _mcolors.to_hex(rgba)
    luminance = 0.299 * rgba[0] + 0.587 * rgba[1] + 0.114 * rgba[2]
    text_color = "black" if luminance > 0.5 else "white"
    return f"background-color: {hex_color}; color: {text_color}"

display(
    df_corr_str_display.style
    .applymap(_cell_bg)
    .set_table_styles([{
        "selector": "th, td",
        "props": "font-family: inherit; font-size: inherit; "
                 "text-align: center; white-space: nowrap; padding: 4px 8px;"
    }])
)

# ── LaTeX table generation for thesis ────────────────────────────────────────
_LATEX_COL_GROUPS = [
    ("Class-Agnostic Detection",
     ["Precision (loc)", "Recall (loc)",
      "Det. Recall 3-way", "Det. Recall 4-way"]),
    ("Joint Combined",
     ["Precision (joint)", "Recall (joint)"]),
    ("Joint 3-way",
     ["Precision 3-way", "Recall 3-way"]),
    ("Joint 4-way",
     ["Precision 4-way", "Recall 4-way"]),
]

_LATEX_COL_HEADERS = {
    "Precision (loc)":   r"$P_{\text{det}}$",
    "Recall (loc)":      r"$R_{\text{det}}$",
    "Det. Recall 3-way": r"$R_{\text{det}}^{\text{3-way}}$",
    "Det. Recall 4-way": r"$R_{\text{det}}^{\text{4-way}}$",
    "Precision (joint)": r"$P_{\text{joint}}$",
    "Recall (joint)":    r"$R_{\text{joint}}$",
    "Precision 3-way":   r"$P_{\text{joint}}^{\text{3-way}}$",
    "Recall 3-way":      r"$R_{\text{joint}}^{\text{3-way}}$",
    "Precision 4-way":   r"$P_{\text{joint}}^{\text{4-way}}$",
    "Recall 4-way":      r"$R_{\text{joint}}^{\text{4-way}}$",
}

def _latex_cell(r):
    if pd.isna(r):
        return "--"
    intensity = int(round((abs(r) ** 1.8) * 75))
    if intensity == 0:
        return f"${r:+.3f}$"
    color = "blue" if r > 0 else "red"
    if intensity >= 50:
        return f"\\cellcolor{{{color}!{intensity}}}\\textcolor{{white}}{{${r:+.3f}$}}"
    return f"\\cellcolor{{{color}!{intensity}}}${r:+.3f}$"

def _build_latex_table(df_corr, n_models):
    ordered_cols = [c for _, cols in _LATEX_COL_GROUPS for c in cols]
    col_spec = "l " + " ".join(
        " ".join(["c"] * len(cols)) for _, cols in _LATEX_COL_GROUPS
    )
    group_header = " & " + " & ".join(
        f"\\multicolumn{{{len(cols)}}}{{c}}{{{name}}}"
        for name, cols in _LATEX_COL_GROUPS
    ) + r" \\"

    cmidrules = []
    start = 2
    for _, cols in _LATEX_COL_GROUPS:
        end = start + len(cols) - 1
        cmidrules.append(f"\\cmidrule(lr){{{start}-{end}}}")
        start = end + 1
    cmidrule_line = " ".join(cmidrules)

    sub_header = " & " + " & ".join(
        _LATEX_COL_HEADERS[c] for c in ordered_cols
    ) + r" \\"

    body_lines = []
    for seg_key in df_corr.index:
        cells = [seg_key] + [_latex_cell(df_corr.loc[seg_key, c])
                             for c in ordered_cols]
        body_lines.append(" & ".join(cells) + r" \\")

    return (
        r"\begin{table}[h]" + "\n"
        r"\centering" + "\n"
        r"\caption{Spearman rank correlation coefficients $r_s$ between "
        r"segmentation metrics and junction detection/classification metrics, "
        f"computed across $n={n_models}$ models from Experiments 1 and 2. "
        r"Cell color encodes correlation strength and direction: blue for "
        r"positive, red for negative, with intensity proportional to $|r_s|$.}"
        + "\n"
        r"\label{tab:spearman_correlations}" + "\n"
        r"\renewcommand{\arraystretch}{1.3}" + "\n"
        r"\setlength{\tabcolsep}{3pt}" + "\n"
        f"\\begin{{tabular}}{{{col_spec}}}" + "\n"
        r"\toprule" + "\n"
        + group_header + "\n"
        + cmidrule_line + "\n"
        + sub_header + "\n"
        r"\midrule" + "\n"
        + "\n".join(body_lines) + "\n"
        r"\bottomrule" + "\n"
        r"\end{tabular}" + "\n"
        r"\end{table}"
    )

print("\n" + "=" * 70)
print("LaTeX table (paste into thesis):")
print("=" * 70 + "\n")
print(_build_latex_table(df_corr, len(_df_corr_input)))


In [ ]:
df_merged.columns

In [ ]:
from matplotlib.lines import Line2D as _Line2D

_INDIVIDUAL_PAIRS = [
    [
        ("tsens", "pp_precision_loc"),
        ("tsens", "pp_class_precision_combined"),
        ("tsens", "pp_class_precision_3way"),
    ],
    [
        ("Wasserstein B0 Raw", "pp_recall_loc"),
        ("Bottleneck B0 Raw", "pp_class_precision_combined"),
        ("Wasserstein B1 Raw", "pp_class_precision_combined"),
        ("Bottleneck B1 Raw", "pp_detection_recall_3way"),
        ("clDice", "pp_detection_recall_4way"),
    ],
    [
        ("Bottleneck B1 SDT", "pp_precision_loc"),
        ("Bottleneck B1 SDT", "pp_class_precision_combined"),
        ("Bottleneck B1 SDT", "pp_class_precision_3way"),
    ],   
]

_FONT = 28

_seg_items = list(SEG_CORR_COLS.items())
_jd_items  = list(ALL_JD_CORR_COLS.items())  # 10 JD metrics -> 2 rows x 5 cols

_N_JD_ROWS, _N_JD_COLS = 2, 5

_GROUP_COLORS = {
    "nnU-Net":                "#e07b39",
    "SAM":                    "#4878d0",
    "SAM (Frozen Encoder)":   "#9467bd",
    "SAM (Baseline Loss)":    "#d62728",
    "SAM (Data Ablation)":    "#6acc65",
}

_FROZEN_ENCODER_MODELS = {"sweep-cldice-BEST", "sweep-skeleton-recall-BEST"}

def _model_group(model):
    m = str(model)
    if "nnunet" in m.lower():
        return "nnU-Net"
    if m in _FROZEN_ENCODER_MODELS:
        return "SAM (Frozen Encoder)"
    if m == "SAM_LoRA_BCE_Dice_Only":
        return "SAM (Baseline Loss)"
    if "SAM_LoRA_Ablation" in m:
        return "SAM (Data Ablation)"
    return "SAM"

_legend_handles = [
    _Line2D([0], [0], marker="o", color="w", markerfacecolor=c,
            markersize=9, label=lbl)
    for lbl, c in _GROUP_COLORS.items()
]

_corr_fig_dir = Path("figures/correlation")
_corr_fig_dir.mkdir(parents=True, exist_ok=True)

_corr_individual_dir = Path("figures/correlation/individual")
_corr_individual_dir.mkdir(parents=True, exist_ok=True)

# ── Individual figures: each nested list becomes one figure ──────────────────
# Each entry is a list of (seg_col, jd_col) pairs (DataFrame column names).
# All pairs in a list are placed side-by-side in one figure with a shared legend.

def _plot_scatter(ax, seg_col, jd_col, font):
    """Draw scatter + OLS onto ax."""
    jd_label = ALL_JD_CORR_COLS.get(jd_col, jd_col)
    seg_label = SEG_CORR_COLS.get(seg_col, seg_col)
    seg_label_clean = seg_label.replace("\n(↓)", " (↓)")
    jd_latex_label = _LATEX_COL_HEADERS.get(jd_label, jd_label)

    valid = df_merged[[seg_col, jd_col]].dropna()
    x = valid[seg_col].values.astype(float)
    y = valid[jd_col].values.astype(float)

    for model, xi, yi in zip(valid.index, x, y):
        grp = _model_group(model)
        ax.scatter(xi, yi, s=90, color=_GROUP_COLORS[grp],
                   zorder=3, alpha=0.85, linewidths=0)

    if len(x) >= 3:
        slope, intercept, *_ = _scipy_stats.linregress(x, y)
        x_fit = np.linspace(x.min(), x.max(), 100)
        ax.plot(x_fit, slope * x_fit + intercept,
                color="black", lw=1.4, alpha=0.45, zorder=2)
        rho, _ = _scipy_stats.spearmanr(x, y)
        ax.set_title(f"$r_s = {rho:+.2f}$", fontsize=font + 1, pad=4)
    else:
        ax.set_title("n < 3", fontsize=font + 1, pad=4)

    ax.set_ylabel(jd_latex_label, fontsize=font)
    ax.set_xlabel(seg_label_clean, fontsize=font)
    ax.tick_params(labelsize=font - 1)
    ax.grid(True, alpha=0.2, linestyle="--")

# ── Main grid plots (one figure per seg metric) ───────────────────────────────
for seg_col, seg_label in _seg_items:
    seg_slug = seg_col.replace(" ", "_").replace("/", "_")

    fig, axes = plt.subplots(_N_JD_ROWS, _N_JD_COLS,
                             figsize=(5.5 * _N_JD_COLS, 4.5 * _N_JD_ROWS),
                             squeeze=False)

    for i, (jd_col, jd_label) in enumerate(_jd_items):
        row, col = i // _N_JD_COLS, i % _N_JD_COLS
        ax = axes[row, col]
        _plot_scatter(ax, seg_col, jd_col, _FONT)
        if row != _N_JD_ROWS - 1:
            ax.set_xlabel("")

    fig.legend(handles=_legend_handles, loc="lower center",
               bbox_to_anchor=(0.5, 1.0), ncol=5,
               fontsize=_FONT, framealpha=0.9)
    plt.tight_layout(rect=[0, 0, 1, 0.97], w_pad=3.0)
    fig.savefig(_corr_fig_dir / f"correlation_{seg_slug}.png",
                dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

# ── Individual figures: one figure per nested list ────────────────────────────
for fig_idx, pairs in enumerate(_INDIVIDUAL_PAIRS):
    valid_pairs = [(s, j) for s, j in pairs
                   if s in df_merged.columns and j in df_merged.columns]
    if not valid_pairs:
        print(f"  Figure {fig_idx}: no valid pairs, skipping.")
        continue

    n = len(valid_pairs)
    fig_ind, axes_ind = plt.subplots(1, n, figsize=(5.5 * n, 4.5), squeeze=False)

    for ax, (seg_col, jd_col) in zip(axes_ind[0], valid_pairs):
        _plot_scatter(ax, seg_col, jd_col, _FONT)

    fig_ind.legend(handles=_legend_handles, loc="lower center",
                   bbox_to_anchor=(0.5, 1.0), ncol=min(n * 2, 5),
                   fontsize=_FONT - 2, framealpha=0.9)
    plt.tight_layout(rect=[0, 0, 1, 0.93], w_pad=3.0)

    # Filename from the pairs in this figure
    slug = "__".join(
        f"{s.replace(' ', '_')}_{j.replace('pp_', '').replace(' ', '_')}"
        for s, j in valid_pairs
    )
    fig_ind.savefig(_corr_individual_dir / f"corr_{fig_idx:02d}.png",
                    dpi=150, bbox_inches="tight")
    plt.close()

print(f"Saved {len(_INDIVIDUAL_PAIRS)} individual figure(s) to {_corr_individual_dir}")
